In [2]:
import os
import zipfile
import random
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile
from keras.preprocessing import image
import numpy as np

In [9]:
def split_data(SOURCE, TRAINING, TESTING, SPLIT_SIZE):
    all_files = []
    
    if len(os.listdir('A:/cats-v-dogs/training/cats/')) != 0:
        print('The necessary files have already been created. Please move to the next cell')
        return
    
    for file_name in os.listdir(SOURCE):
        file_path = SOURCE + file_name
        
        if os.path.getsize(file_path):
            all_files.append(file_name)
        else:
            print('{} is zero length, so ignoring'.format(file_name))
    
    n_files = len(all_files)
    split_point = int(n_files * SPLIT_SIZE)
    
    shuffled = random.sample(all_files, n_files)
    
    train_set = shuffled[:split_point]
    test_set = shuffled[split_point:]
    
    for file_name in train_set:
        copyfile(SOURCE + file_name, TRAINING + file_name)
        
    for file_name in test_set:
        copyfile(SOURCE + file_name, TESTING + file_name)

#I HAVE HARDCODED THE DIRECTORY. CHANGE WHEN MOVE FILE AROUND
CAT_SOURCE_DIR = 'A:/cats-v-dogs-extracted/Cat/'
TRAINING_CATS_DIR = "A:/cats-v-dogs/training/cats/"
TESTING_CATS_DIR = "A:/cats-v-dogs/testing/cats/"
DOG_SOURCE_DIR = "A:/cats-v-dogs-extracted/Dog/"
TRAINING_DOGS_DIR = "A:/cats-v-dogs/training/dogs/"
TESTING_DOGS_DIR = "A:/cats-v-dogs/testing/dogs/"

split_size = .9
split_data(CAT_SOURCE_DIR, TRAINING_CATS_DIR, TESTING_CATS_DIR, split_size)
split_data(DOG_SOURCE_DIR, TRAINING_DOGS_DIR, TESTING_DOGS_DIR, split_size)

# Expected output
# 666.jpg is zero length, so ignoring
# 11702.jpg is zero length, so ignoring

The necessary files have already been created. /n Please move to the next cell
The necessary files have already been created. /n Please move to the next cell


In [6]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation = 'relu', input_shape = (150,150, 3)),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation = 'relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation = 'relu'),
    tf.keras.layers.Dense(1, activation = 'sigmoid')
])

model.compile(optimizer=RMSprop(lr=0.001), metrics=['acc'], loss= 'binary_crossentropy')

train_datagen = ImageDataGenerator(
    rescale = 1/255,
    horizontal_flip = True,
    rotation_range=40,
    width_shift_range=.2,
    height_shift_range=.2,
    shear_range=.2,
    zoom_range=.2,
    fill_mode = 'nearest'              
)

#LENGTH OF TRAIN AND TEST DATA = 11250, 1250
train_generator = train_datagen.flow_from_directory(
    'A:/cats-v-dogs/training/',
    batch_size = 64,
    class_mode = 'binary',
    target_size = (150,150)    
)

validation_datagen = ImageDataGenerator(
    rescale=1 / 255,
    rotation_range=40,
    width_shift_range=.2,
    height_shift_range=.2,
    shear_range=.2,
    zoom_range=.2,
    horizontal_flip=True,
    fill_mode='nearest'

)
validation_generator = validation_datagen.flow_from_directory(
    'A:/cats-v-dogs/testing/',
    batch_size=64,
    class_mode='binary',
    target_size=(150, 150)
)




Found 22499 images belonging to 2 classes.
Found 2499 images belonging to 2 classes.


In [7]:
#Change the number of epochs for better accuracy

import warnings
warnings.filterwarnings('ignore')
print('warnings ignored')

history = model.fit_generator(train_generator,
                              epochs=1,
                              verbose=1,
                              validation_data=validation_generator)

warnings ignored
Instructions for updating:
Please use Model.fit, which supports generators.
352/352 [==============================] - 17653s 50s/step - loss: 0.7125 - acc: 0.5938 - val_loss: 0.6069 - val_acc: 0.6839


In [5]:
from PIL import Image  
from IPython.display import Image, display

#CHANGE PATH TO CLASSIFY DIFFERENT PICTURES
img_path = '/dog.jpg'
img = image.load_img(img_path, target_size=(150,150,3))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)

images = np.vstack([x])
classes = model.predict(images, batch_size=10)
print(classes[0])
if classes[0]>0.5:
    print("The given image is a dog")
else:
    print("The given image is a cat")

display(Image(filename= img_path))

FileNotFoundError: [Errno 2] No such file or directory: 'A:/155.jpg'